# Music highlight / chorus / segment-selection model benchmark

Quick-and-dirty inference benchmark comparing approaches for **automatic highlight/chorus
detection** -- picking the single most representative ~10-30s window of a song -- aimed at
finding something light + fast enough to run **on-device on iOS** (same spirit as
`../music_retrieval_comparison/compare_music_retrieval_models.ipynb` and
`../notebooks/benchmark_clip_models.ipynb`).

This is the model that would fill in `ResultView`'s "Track Segment" placeholder (currently
a hardcoded `0:45-1:15`) with a real, per-song highlight window.

## The task

Given a full song, return a single `(start, end)` timestamp range (target: ~10-30s,
matching the app's existing placeholder length) that best represents the song's
"highlight" -- in practice, usually the most-repeated *and* highest-energy section
(typically the chorus/hook).

## Candidates

| # | Approach | Family | Trained params |
|---|---|---|---|
| 1 | Self-similarity novelty (Foote, chroma-CQT) | classical DSP | 0 |
| 2 | Laplacian structural segmentation (McFee & Ellis) | classical DSP + clustering | 0 |
| 3 | MuQ (SSL audio embeddings) self-similarity | deep learning (frozen backbone) | ~333M |
| 4 | All-In-One Music Structure Analyzer (`allin1`) | deep learning, trained end-to-end for this exact task | n/a (fails to load here) |

Candidates 1 and 2 need **no model file at all** -- they're pure signal processing and
could ship as Swift/Accelerate code with zero CoreML conversion. Candidate 3 reuses the
SSL backbone this project already vetted for retrieval (`music_retrieval_comparison/`)
and repurposes it for structure via the same self-similarity idea as candidate 1, at
~300M-parameter cost. Candidate 4 is the closest thing to a dedicated, published SOTA
model for this exact task (ISMIR 2023 LBD) -- see the markdown note near its cell for
why it's kept in the comparison but doesn't actually run in this environment.

In [1]:
import statistics
import time
import warnings
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf
import torch

warnings.filterwarnings("ignore")

In [2]:
AUDIO_DIR = Path.cwd().parent / "audios"
OUTPUT_CLIPS_DIR = Path.cwd() / "output_clips"

SR = 22050  # shared sample rate for the DSP-based candidates (chroma / CQT / beat-tracking / RMS)

MIN_HIGHLIGHT_S = 10.0     # shorter than this isn't a usable "highlight" preview
MAX_HIGHLIGHT_S = 30.0     # matches ResultView's existing "Track Segment" placeholder length (0:45-1:15)
TARGET_HIGHLIGHT_S = 30.0  # trim longer candidate segments down to this

NOVELTY_RADIUS_S = 8.0      # checkerboard-kernel half-width -> detects section-scale boundaries, not beat/bar-scale
MIN_BOUNDARY_GAP_S = 10.0   # minimum spacing enforced between detected boundaries

## Helpers

Shared utilities used by more than one candidate:
- `load_track(path)` decodes a track to mono at `SR`.
- `checkerboard_kernel` / `novelty_curve` / `boundary_indices_from_ssm` implement Foote's
  classic self-similarity novelty method generically over *any* per-frame (or per-window)
  feature matrix -- used by both the chroma-based and the MuQ-embedding-based candidates.
- `score_and_pick_segment` turns a list of `(t0, t1)` segments + per-segment feature
  vectors into a single highlight pick: score = *how much this segment repeats elsewhere*
  x *how high-energy it is*, since a chorus is usually both.
- `state_dict_size_mb` / `fmt_time` mirror the retrieval notebook's helpers.

In [3]:
def load_track(path, sr=SR):
    y, _ = librosa.load(path, sr=sr, mono=True)
    return y


def fmt_time(seconds):
    m, s = divmod(int(round(seconds)), 60)
    return f"{m}:{s:02d}"


def state_dict_size_mb(module):
    import io

    buf = io.BytesIO()
    torch.save(module.state_dict(), buf)
    return buf.getbuffer().nbytes / 1e6


def checkerboard_kernel(size, sigma_ratio=0.4):
    axis = np.arange(-size, size + 1)
    gauss = np.exp(-(axis**2) / (2 * (size * sigma_ratio) ** 2))
    taper = np.outer(gauss, gauss)
    sign = np.outer(np.sign(axis + 1e-9), np.sign(axis + 1e-9))
    return sign * taper


def novelty_curve(ssm, kernel_size):
    kernel = checkerboard_kernel(kernel_size)
    n = ssm.shape[0]
    novelty = np.zeros(n)
    for t in range(kernel_size, n - kernel_size):
        window = ssm[t - kernel_size : t + kernel_size + 1, t - kernel_size : t + kernel_size + 1]
        novelty[t] = np.sum(window * kernel)
    novelty -= novelty.min()
    if novelty.max() > 0:
        novelty /= novelty.max()
    return novelty


def boundary_indices_from_ssm(ssm, kernel_radius, min_gap_steps):
    """Foote's checkerboard-kernel novelty method: peaks in the novelty curve = section boundaries."""
    n = ssm.shape[0]
    kernel_radius = max(1, min(kernel_radius, (n - 1) // 2))
    if n < 2 * kernel_radius + 2:
        return np.array([0, n - 1])
    novelty = novelty_curve(ssm, kernel_radius)
    min_gap_steps = max(1, min_gap_steps)
    peaks = librosa.util.peak_pick(
        novelty,
        pre_max=min_gap_steps, post_max=min_gap_steps,
        pre_avg=min_gap_steps, post_avg=min_gap_steps,
        delta=0.02, wait=min_gap_steps,
    )
    return np.unique(np.concatenate([[0], peaks, [n - 1]]))


def score_and_pick_segment(segments, seg_features, y, sr, min_dur=MIN_HIGHLIGHT_S,
                            max_dur=MAX_HIGHLIGHT_S, target_dur=TARGET_HIGHLIGHT_S):
    """segments: list of (t0, t1). seg_features: matching list of mean feature vectors.
    Picks the segment that is both most repeated elsewhere (cosine similarity to its
    best-matching other segment) and highest-energy (RMS) -- the two properties that
    make a chorus a chorus."""
    feats = np.stack(seg_features)
    feats = feats / (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-9)
    sim = feats @ feats.T
    np.fill_diagonal(sim, -1.0)

    rms = librosa.feature.rms(y=y)[0]
    rms_times = librosa.frames_to_time(np.arange(len(rms)), sr=sr)

    rms_vals = []
    for t0, t1 in segments:
        mask = (rms_times >= t0) & (rms_times < t1)
        rms_vals.append(float(rms[mask].mean()) if mask.any() else 0.0)
    rms_vals = np.array(rms_vals)
    rms_norm = (rms_vals - rms_vals.min()) / (rms_vals.max() - rms_vals.min() + 1e-9)

    scored = []
    for i, (t0, t1) in enumerate(segments):
        repetition = float(sim[i].max()) if len(segments) > 1 else 0.0
        energy = float(rms_norm[i])
        scored.append({
            "t0": t0, "t1": t1, "dur": t1 - t0,
            "repetition": repetition, "energy": energy,
            "score": repetition * (0.5 + 0.5 * energy),
        })

    candidates = [s for s in scored if s["dur"] >= min_dur]
    best = max(candidates, key=lambda s: s["score"]) if candidates else max(scored, key=lambda s: s["dur"])
    t0, t1 = best["t0"], best["t1"]
    if t1 - t0 > max_dur:
        t1 = t0 + target_dur
    return t0, t1, scored

## Candidate adapters

Each `load_*()` returns `(analyze_fn, n_params, size_mb)`:
- `analyze_fn(path)` -> `(start_s, end_s, debug_segments)` for a *full* track (not a
  truncated clip -- structure detection needs the whole song, unlike the retrieval
  notebook's fixed 30s clips).
- `n_params` / `size_mb` describe the trained weights that would need to ship in the iOS
  bundle (`0` for the two pure-DSP candidates).

Loading (including any imports) happens lazily inside each `load_*()` so a candidate
that fails to install/import in a given environment fails gracefully at benchmark time
instead of at notebook-load time -- same pattern the retrieval notebook uses (see
`allin1` below for a real example of this).

In [4]:
def load_foote_chroma():
    """Classical DSP baseline: chroma self-similarity + Foote novelty boundaries,
    then pick the most-repeated/highest-energy segment. Zero trained parameters."""

    def analyze(path):
        y = load_track(path)
        hop_length = 2048
        chroma = librosa.feature.chroma_cqt(y=y, sr=SR, hop_length=hop_length)
        chroma = librosa.util.normalize(chroma, axis=0)
        ssm = chroma.T @ chroma

        kernel_radius = max(2, int(round(NOVELTY_RADIUS_S * SR / hop_length)))
        min_gap_steps = int(round(MIN_BOUNDARY_GAP_S * SR / hop_length))
        bound_idx = boundary_indices_from_ssm(ssm, kernel_radius, min_gap_steps)

        bound_times = librosa.frames_to_time(bound_idx, sr=SR, hop_length=hop_length)
        bound_times[-1] = len(y) / SR  # snap last boundary to the true track end

        segments = [(bound_times[i], bound_times[i + 1]) for i in range(len(bound_idx) - 1)]
        seg_features = [
            chroma[:, bound_idx[i] : bound_idx[i + 1]].mean(axis=1)
            for i in range(len(bound_idx) - 1)
        ]
        return score_and_pick_segment(segments, seg_features, y, SR)

    return analyze, 0, 0.0

In [5]:
def load_laplacian_segmentation():
    """Classical structural-features segmentation (McFee & Ellis, 2014): beat-synchronous
    recurrence + local-path affinity, spectral (Laplacian eigenmap) clustering into a small
    number of section-type labels, then pick the label that recurs most and covers the most
    of the song, weighted by energy. Zero trained parameters."""
    import scipy
    import sklearn.cluster
    from collections import defaultdict

    def mode_smooth(seg_ids, window=9):
        """Rolling-window majority vote to remove isolated 1-2 beat cluster flickers."""
        n = len(seg_ids)
        half = window // 2
        out = seg_ids.copy()
        for i in range(n):
            lo, hi = max(0, i - half), min(n, i + half + 1)
            vals, counts = np.unique(seg_ids[lo:hi], return_counts=True)
            out[i] = vals[np.argmax(counts)]
        return out

    def merge_short_segments(segments, min_seg_dur=6.0):
        """Fold segments shorter than min_seg_dur into their larger neighbor (beat-tracking
        flicker cleanup) so "sections" are actually section-length."""
        segs = [dict(s) for s in segments]
        changed = True
        while changed and len(segs) > 1:
            changed = False
            for i, s in enumerate(segs):
                if s["dur"] >= min_seg_dur:
                    continue
                neighbors = [j for j in (i - 1, i + 1) if 0 <= j < len(segs)]
                j = max(neighbors, key=lambda j: segs[j]["dur"])
                lo, hi = min(i, j), max(i, j)
                merged = {"t0": segs[lo]["t0"], "t1": segs[hi]["t1"], "label": segs[j]["label"]}
                merged["dur"] = merged["t1"] - merged["t0"]
                segs[lo : hi + 1] = [merged]
                changed = True
                break
        return segs

    def analyze(path):
        y = load_track(path)
        sr = SR
        bins_per_octave, n_octaves = 12 * 3, 7
        C = librosa.amplitude_to_db(
            np.abs(librosa.cqt(y=y, sr=sr, bins_per_octave=bins_per_octave,
                                n_bins=n_octaves * bins_per_octave)),
            ref=np.max,
        )
        _, beats = librosa.beat.beat_track(y=y, sr=sr, trim=False)
        beats = librosa.util.fix_frames(beats, x_min=0)
        Csync = librosa.util.sync(C, beats, aggregate=np.median)

        R = librosa.segment.recurrence_matrix(Csync, width=3, mode="affinity", sym=True)
        Rf = librosa.segment.timelag_filter(scipy.ndimage.median_filter)(R, size=(1, 7))

        mfcc = librosa.feature.mfcc(y=y, sr=sr)
        Msync = librosa.util.sync(mfcc, beats)
        path_distance = np.sum(np.diff(Msync, axis=1) ** 2, axis=0)
        sigma = np.median(path_distance) + 1e-9
        path_sim = np.exp(-path_distance / sigma)
        R_path = np.diag(path_sim, k=1) + np.diag(path_sim, k=-1)

        deg_path, deg_rec = np.sum(R_path, axis=1), np.sum(Rf, axis=1)
        mu = deg_path.dot(deg_path + deg_rec) / np.sum((deg_path + deg_rec) ** 2)
        A = mu * Rf + (1 - mu) * R_path

        L = scipy.sparse.csgraph.laplacian(A, normed=True)
        _, evecs = scipy.linalg.eigh(L)
        evecs = scipy.ndimage.median_filter(evecs, size=(9, 1))
        cnorm = np.cumsum(evecs**2, axis=1) ** 0.5

        k = min(6, evecs.shape[1])
        X = evecs[:, :k] / (cnorm[:, k - 1 : k] + 1e-8)
        seg_ids = sklearn.cluster.KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
        seg_ids = mode_smooth(seg_ids, window=9)

        bound_beats = librosa.util.fix_frames(1 + np.flatnonzero(seg_ids[:-1] != seg_ids[1:]), x_min=0)
        bound_frames = librosa.util.fix_frames(beats[bound_beats], x_min=None, x_max=C.shape[1] - 1)
        bound_times = librosa.frames_to_time(bound_frames, sr=sr)
        bound_times[-1] = len(y) / sr
        bound_segs = seg_ids[bound_beats[:-1]] if len(bound_beats) > 1 else seg_ids

        segments = [
            {"t0": bound_times[i], "t1": bound_times[i + 1], "label": int(bound_segs[i]),
             "dur": bound_times[i + 1] - bound_times[i]}
            for i in range(len(bound_segs))
        ]
        segments = merge_short_segments(segments, min_seg_dur=6.0)

        label_total_dur, label_occurrences = defaultdict(float), defaultdict(int)
        for s in segments:
            label_total_dur[s["label"]] += s["dur"]
            label_occurrences[s["label"]] += 1

        rms = librosa.feature.rms(y=y)[0]
        rms_times = librosa.frames_to_time(np.arange(len(rms)), sr=sr)
        rms_vals = []
        for s in segments:
            mask = (rms_times >= s["t0"]) & (rms_times < s["t1"])
            rms_vals.append(float(rms[mask].mean()) if mask.any() else 0.0)
        rms_vals = np.array(rms_vals)
        rms_norm = (rms_vals - rms_vals.min()) / (rms_vals.max() - rms_vals.min() + 1e-9)

        scored = []
        for s, energy in zip(segments, rms_norm):
            # reward labels that recur (>=2 occurrences) and cover a lot of the song
            repetition = label_total_dur[s["label"]] * min(label_occurrences[s["label"]], 4)
            scored.append({**s, "energy": float(energy), "repetition": repetition,
                            "score": repetition * (0.5 + 0.5 * float(energy))})

        candidates = [s for s in scored if s["dur"] >= MIN_HIGHLIGHT_S]
        best = max(candidates, key=lambda s: s["score"]) if candidates else max(scored, key=lambda s: s["dur"])
        t0, t1 = best["t0"], best["t1"]
        if t1 - t0 > MAX_HIGHLIGHT_S:
            t1 = t0 + TARGET_HIGHLIGHT_S
        return t0, t1, scored

    return analyze, 0, 0.0

In [6]:
MUQ_MODEL_ID = "OpenMuQ/MuQ-large-msd-iter"
MUQ_SR = 24000
MUQ_WINDOW_S = 10.0
MUQ_HOP_S = 5.0


def load_muq_selfsim():
    """Reuses the MuQ self-supervised audio backbone already vetted in
    ../music_retrieval_comparison/ for retrieval, repurposed for structure: embed
    overlapping 10s windows, then run the exact same novelty + repetition/energy
    scoring as the chroma baseline, but on *learned* embeddings instead of chroma."""
    import muq

    model = muq.MuQ.from_pretrained(MUQ_MODEL_ID)
    model.eval()
    n_params = sum(p.numel() for p in model.parameters())
    size_mb = state_dict_size_mb(model)

    def embed_windows(y):
        y24k = librosa.resample(y, orig_sr=SR, target_sr=MUQ_SR) if SR != MUQ_SR else y
        win, hop = int(MUQ_WINDOW_S * MUQ_SR), int(MUQ_HOP_S * MUQ_SR)
        starts = list(range(0, max(1, len(y24k) - win), hop)) or [0]
        embs, times = [], []
        with torch.no_grad():
            for s in starts:
                chunk = y24k[s : s + win]
                if len(chunk) < win:
                    chunk = np.pad(chunk, (0, win - len(chunk)))
                wav = torch.from_numpy(chunk).float().unsqueeze(0)
                out = model(wav, output_hidden_states=False)
                embs.append(out.last_hidden_state.mean(dim=1).squeeze(0).numpy())
                times.append(s / MUQ_SR)
        return np.stack(embs), np.array(times)

    def analyze(path):
        y = load_track(path)
        embs, win_times = embed_windows(y)
        embs_n = embs / (np.linalg.norm(embs, axis=1, keepdims=True) + 1e-9)
        ssm = embs_n @ embs_n.T

        kernel_radius = max(1, int(round(1.5 * MUQ_WINDOW_S / MUQ_HOP_S)))
        min_gap_steps = max(1, int(round(MIN_BOUNDARY_GAP_S / MUQ_HOP_S)))
        bound_idx = boundary_indices_from_ssm(ssm, kernel_radius, min_gap_steps)

        bound_times = list(win_times[bound_idx[:-1]]) + [len(y) / SR]
        segments = [(bound_times[i], bound_times[i + 1]) for i in range(len(bound_idx) - 1)]
        seg_features = [
            embs_n[bound_idx[i] : bound_idx[i + 1]].mean(axis=0) for i in range(len(bound_idx) - 1)
        ]
        return score_and_pick_segment(segments, seg_features, y, SR)

    return analyze, n_params, size_mb

### About `all-in-one` (`allin1`)

[`allin1`](https://github.com/mir-aidj/all-in-one) is the closest thing to a dedicated,
published SOTA model for this exact task: a Demucs front-end + DiNAT (dilated
neighborhood-attention) transformer trained specifically to output
`verse`/`chorus`/`bridge`/... functional segment labels. It's kept in the comparison
below via the same try/except pattern the retrieval notebook uses for candidates that
don't pan out in a given environment, rather than silently dropped.

In this environment it fails to *import*: `allin1`'s DiNAT layers call
`natten.functional.natten1dav` / `natten1dqkrpb`, functions that were removed when
`natten` unified its API around `na1d`/`na2d` in v0.15+, and `allin1` doesn't pin an
upper bound on `natten`, so `pip`/`uv` resolve its latest release. Pinning an older,
API-compatible `natten` release (`natten==0.17.5`) isn't a clean fix either -- that
version's `setup.py` doesn't request the C++20 build config it needs, and fails to
compile even with a C++20-capable compiler and `cmake` present. Worth revisiting if
`allin1` or `natten` fix this upstream, or if it gets vendored/patched locally.

In [7]:
def load_allin1():
    import allin1  # see markdown note above -- fails to import in this environment

    def analyze(path):
        result = allin1.analyze(str(path), keep_byproducts=False, overwrite=True)
        chorus_segs = [s for s in result.segments if s.label == "chorus"]
        candidates = chorus_segs or list(result.segments)
        best = max(candidates, key=lambda s: s.end - s.start)
        t0, t1 = best.start, best.end
        if t1 - t0 > MAX_HIGHLIGHT_S:
            t1 = t0 + TARGET_HIGHLIGHT_S
        return t0, t1, result.segments

    return analyze, float("nan"), float("nan")

In [8]:
MODEL_SPECS = [
    {"label": "Self-similarity novelty (Foote, chroma-CQT)", "loader": load_foote_chroma},
    {"label": "Laplacian structural segmentation (McFee & Ellis)", "loader": load_laplacian_segmentation},
    {"label": "MuQ SSL-embedding self-similarity", "loader": load_muq_selfsim},
    {"label": "All-In-One (allin1, Demucs + DiNAT transformer)", "loader": load_allin1},
]

## Benchmark

For each candidate: load (report params + fp32 size), then run full-track inference on
every song in `../audios`, timing each track individually -- structure detection needs
the whole track, so unlike the retrieval notebook's fixed 30s batches, there's no short
clip to repeat-and-average timing over.

In [9]:
track_paths = sorted(p for p in AUDIO_DIR.iterdir() if p.suffix.lower() == ".mp3")
if not track_paths:
    raise FileNotFoundError(f"No audio files found in {AUDIO_DIR}")


def run_analysis(spec):
    print(f"=== {spec['label']} ===")
    t0 = time.perf_counter()
    try:
        analyze, n_params, size_mb = spec["loader"]()
    except Exception as e:
        print(f"  FAILED to load: {e}\n")
        return None
    load_s = time.perf_counter() - t0
    print(f"  params: {n_params / 1e6:.2f}M | fp32 size: {size_mb:.1f}MB | load time: {load_s:.1f}s")

    per_track = []
    for path in track_paths:
        t0 = time.perf_counter()
        try:
            start, end, _debug = analyze(path)
        except Exception as e:
            print(f"    {path.stem:32s} FAILED: {e}")
            continue
        infer_s = time.perf_counter() - t0
        per_track.append({"path": path, "start": start, "end": end, "infer_s": infer_s})
        print(
            f"    {path.stem:32s} highlight {fmt_time(start)}-{fmt_time(end)} "
            f"({end - start:4.1f}s)   infer={infer_s:6.2f}s"
        )

    if not per_track:
        print()
        return None

    mean_infer_s = statistics.mean(p["infer_s"] for p in per_track)
    print(f"  mean inference time per full track: {mean_infer_s:.2f}s\n")

    return {
        "label": spec["label"], "params_m": n_params / 1e6, "size_mb": size_mb,
        "load_s": load_s, "mean_infer_s": mean_infer_s, "per_track": per_track,
    }


results = [r for r in (run_analysis(spec) for spec in MODEL_SPECS) if r is not None]

=== Self-similarity novelty (Foote, chroma-CQT) ===
  params: 0.00M | fp32 size: 0.0MB | load time: 0.0s


    Cheek To Cheek                   highlight 2:59-3:12 (12.9s)   infer=  1.35s


    Cherry Wine                      highlight 2:10-2:26 (16.0s)   infer=  0.38s


    Clementine                       highlight 2:02-2:31 (28.4s)   infer=  0.63s


    From The Start                   highlight 2:16-2:30 (13.9s)   infer=  0.40s


...

    Happy!                           highlight 2:05-2:20 (15.0s)   infer= 24.59s


    I Hope to Be Around              highlight 2:35-2:50 (15.0s)   infer= 30.74s


    Numb                             highlight 2:45-3:10 (25.0s)   infer= 31.89s


    Regent's Park                    highlight 1:35-2:05 (30.0s)   infer= 25.74s


    The Most Beautiful Thing         highlight 2:45-3:15 (30.0s)   infer= 23.16s
  mean inference time per full track: 29.02s

=== All-In-One (allin1, Demucs + DiNAT transformer) ===
  FAILED to load: cannot import name 'natten1dav' from 'natten.functional' (/Users/rafiabhista/Documents/code/apple/c3_agentic_ai/app-v1/trial_models/.venv/lib/python3.12/site-packages/natten/functional.py)



## Summary

In [10]:
header = f"{'Model':50s} {'Params(M)':>10s} {'FP32(MB)':>9s} {'Load(s)':>8s} {'Infer(s/track)':>15s}"
print(header)
for r in results:
    print(f"{r['label']:50s} {r['params_m']:10.2f} {r['size_mb']:9.1f} {r['load_s']:8.1f} {r['mean_infer_s']:15.2f}")

Model                                               Params(M)  FP32(MB)  Load(s)  Infer(s/track)
Self-similarity novelty (Foote, chroma-CQT)              0.00       0.0      0.0            0.56
Laplacian structural segmentation (McFee & Ellis)        0.00       0.0      0.2            0.75
MuQ SSL-embedding self-similarity                      333.40    1334.0      6.9           29.02


## Test on real music

Per-track highlight picks, side by side across every candidate that loaded
successfully -- read these before trusting the numbers above, and skim-listen with the
clip-export cell further down.

In [11]:
col_w = max(len(r["label"]) for r in results) + 2
header = f"{'Track':32s}" + "".join(f"{r['label']:<{col_w}s}" for r in results)
print(header)
for path in track_paths:
    row = f"{path.stem:32s}"
    for r in results:
        entry = next((p for p in r["per_track"] if p["path"] == path), None)
        cell = f"{fmt_time(entry['start'])}-{fmt_time(entry['end'])}" if entry else "n/a"
        row += f"{cell:<{col_w}s}"
    print(row)

Track                           Self-similarity novelty (Foote, chroma-CQT)        Laplacian structural segmentation (McFee & Ellis)  MuQ SSL-embedding self-similarity                  
Cheek To Cheek                  2:59-3:12                                          1:24-1:54                                          1:10-1:40                                          
Cherry Wine                     2:10-2:26                                          2:04-2:34                                          1:55-2:25                                          
Clementine                      2:02-2:31                                          3:04-3:34                                          3:00-3:30                                          
From The Start                  2:16-2:30                                          1:42-2:03                                          2:05-2:20                                          
Happy!                          1:50-2:07                             

## Export highlight clips for listening

The numbers above can't tell you whether a "highlight" actually *sounds* like the hook.
This writes each candidate's pick to a `.wav` for a quick skim-listen:
`output_clips/<candidate>/<track>.wav`.

In [12]:
import re


def slugify(text):
    return re.sub(r"[^a-z0-9]+", "-", text.lower()).strip("-")


for r in results:
    out_dir = OUTPUT_CLIPS_DIR / slugify(r["label"])
    out_dir.mkdir(parents=True, exist_ok=True)
    for entry in r["per_track"]:
        y, sr = librosa.load(entry["path"], sr=None, mono=True)
        i0, i1 = int(entry["start"] * sr), int(entry["end"] * sr)
        sf.write(out_dir / f"{entry['path'].stem}.wav", y[i0:i1], sr)

print(f"Wrote clips to {OUTPUT_CLIPS_DIR}")

Wrote clips to /Users/rafiabhista/Documents/code/apple/c3_agentic_ai/app-v1/trial_models/music_highlight_comparison/output_clips


## Recommendation

Ranked by on-disk size -- the dominant constraint for shipping in the iOS app bundle,
since that's what would need to become a `.mlpackage`:

In [13]:
ranked = sorted(results, key=lambda r: r["size_mb"])
print(f"{'Model':50s} {'Size(MB)':>9s} {'Infer(s/track)':>15s}")
for r in ranked:
    print(f"{r['label']:50s} {r['size_mb']:9.1f} {r['mean_infer_s']:15.2f}")

Model                                               Size(MB)  Infer(s/track)
Self-similarity novelty (Foote, chroma-CQT)              0.0            0.56
Laplacian structural segmentation (McFee & Ellis)        0.0            0.75
MuQ SSL-embedding self-similarity                     1334.0           29.02


Both DSP candidates ship at **zero model size** (pure signal processing -- chroma/CQT,
beat tracking, RMS -- all doable directly in Accelerate/vDSP, no CoreML conversion
needed) and run a full track in well under a second. MuQ gets a *learned* notion of
"this sounds similar" instead of hand-crafted chroma, which can produce tighter section
boundaries, at the cost of a ~330M-parameter backbone that is nowhere near mobile-sized
as-is (would need real distillation, not just quantization, to be iOS-viable).

Given the constraint is "light and usable locally," the two DSP candidates are the
practical choice today. `allin1` is the one worth revisiting once its `natten`
dependency is fixed upstream (or vendored/patched locally) -- it's the only candidate
here actually trained end-to-end for this task rather than repurposing a similarity
heuristic, and it would need a genuinely different iOS story anyway (Demucs source
separation is itself a heavy model).